# Generalisation table — PCA vs AE, train vs held-out

Builds the **5-fold spatially blocked cross-validation** table used in §3.1: reconstruction RMSE
for PCA and the autoencoder, each evaluated on the fold it trained on (**train**) and on the
held-out LAD fold (**held-out**), as mean (min–max) across the five folds of one LAD shuffle.

* AE errors are read directly from the notebook 2c (`2c_lad_blocked_cv`) checkpoints (`train_rmse`, `test_rmse` per fold).
* PCA per-fold errors are read from `pca_spcv_results.pkl`, also saved by notebook 2c.

Exports the table to `tables/cv_2x2_rmse.{csv,html,tex}` for the paper / Word.

In [1]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

bottleneck_sizes = [2, 4, 8, 16, 32, 64, 100, 128]
repeat  = 0            # single LAD shuffle
n_folds = 5
force_recompute_pca = False

LAD_DIR      = "../AE_outputs/retraining_stability_500epochs_linscaling_lad_blocked"
DATA_PARQUET = "../data/census_data/engcensus_cleaned_scaled.parquet"
TABLE_DIR    = "tables"
os.makedirs(TABLE_DIR, exist_ok=True)

## 1. AE train & held-out RMSE from the 2c checkpoints

In [2]:
ae_train, ae_test = [], []   # per-dim arrays of the 5 fold RMSEs (%)
for d in bottleneck_sizes:
    with open(f"{LAD_DIR}/data/spcv_checkpoint_{d}d_rep{repeat}.pkl", 'rb') as f:
        ck = pickle.load(f)
    assert ck.get('n_folds_done', 0) >= n_folds, f"{d}d rep{repeat} incomplete"
    ae_train.append(np.array([ck['folds'][k]['train_rmse'] for k in range(n_folds)]) * 100)
    ae_test.append(np.array([ck['folds'][k]['test_rmse']  for k in range(n_folds)]) * 100)
print("Loaded AE train/held-out across", n_folds, "folds")

Loaded AE train/held-out across 5 folds


## 2. PCA train & held-out RMSE on the same folds

Per-fold PCA errors are read from `pca_spcv_results.pkl`, saved by the `2c_lad_blocked_cv` notebook when it fit
PCA inside each spatial fold — no refit needed here. (The old `pca_cv_traintest_rep*.pkl` cache only
stored means/SDs, not per-fold values; if present it is used as a consistency check.)

In [3]:
with open(f"{LAD_DIR}/data/pca_spcv_results.pkl", 'rb') as f:
    _pf = pickle.load(f)['per_fold']

pca_train = [np.array([_pf[(d, repeat, k)]['train_rmse'] for k in range(n_folds)]) * 100
             for d in bottleneck_sizes]
pca_test  = [np.array([_pf[(d, repeat, k)]['test_rmse']  for k in range(n_folds)]) * 100
             for d in bottleneck_sizes]

# consistency check against the old means-only cache, if it exists
pca_cache = f"{LAD_DIR}/data/pca_cv_traintest_rep{repeat}.pkl"
if os.path.exists(pca_cache):
    with open(pca_cache, 'rb') as f:
        old = pickle.load(f)
    assert np.allclose([v.mean() for v in pca_test], old['test_m'], atol=1e-9), \
        "per-fold PCA results disagree with cached means"
    print("Per-fold PCA loaded; means match the old cache")
else:
    print("Per-fold PCA loaded")

Per-fold PCA loaded; means match the old cache


## 3. Assemble and display the table

In [4]:
def pmm(v):  # mean (min–max) across the five folds — held-out columns
    return f"{v.mean():.3f} ({v.min():.3f}–{v.max():.3f})"

table = pd.DataFrame({
    'Dimensionality': bottleneck_sizes,
    'PCA (train)':     [f"{v.mean():.3f}" for v in pca_train],
    'PCA (held-out)':  [pmm(v) for v in pca_test],
    'AE (train)':      [f"{v.mean():.3f}" for v in ae_train],
    'AE (held-out)':   [pmm(v) for v in ae_test],
})

# upper bound on train fold-to-fold spread, quoted in the caption
train_spread_bound = max((v.max() - v.min()) for v in pca_train + ae_train)
print(f"max train fold spread: {train_spread_bound:.3f} pp")
print(table.to_string(index=False))
table

max train fold spread: 0.094 pp
 Dimensionality PCA (train)      PCA (held-out) AE (train)       AE (held-out)
              2       5.445 5.465 (5.335–5.539)      3.909 4.071 (3.999–4.114)
              4       4.565 4.580 (4.498–4.677)      3.134 3.328 (3.273–3.385)
              8       3.720 3.744 (3.683–3.846)      2.511 2.648 (2.593–2.681)
             16       2.813 2.836 (2.744–2.897)      2.003 2.084 (2.024–2.112)
             32       2.033 2.055 (1.970–2.110)      1.562 1.607 (1.560–1.634)
             64       1.359 1.380 (1.317–1.413)      1.105 1.134 (1.098–1.165)
            100       0.944 0.964 (0.915–0.989)      0.787 0.813 (0.770–0.870)
            128       0.723 0.745 (0.704–0.780)      0.685 0.708 (0.665–0.792)


,Dimensionality,PCA (train),PCA (held-out),AE (train),AE (held-out)
0,2,5.445,5.465 (5.335–5.539),3.909,4.071 (3.999–4.114)
1,4,4.565,4.580 (4.498–4.677),3.134,3.328 (3.273–3.385)
2,8,3.720,3.744 (3.683–3.846),2.511,2.648 (2.593–2.681)
3,16,2.813,2.836 (2.744–2.897),2.003,2.084 (2.024–2.112)
4,32,2.033,2.055 (1.970–2.110),1.562,1.607 (1.560–1.634)
5,64,1.359,1.380 (1.317–1.413),1.105,1.134 (1.098–1.165)
6,100,0.944,0.964 (0.915–0.989),0.787,0.813 (0.770–0.870)
7,128,0.723,0.745 (0.704–0.780),0.685,0.708 (0.665–0.792)


## 3b. Check — CV train means vs full-sample fits

Supports the §3.1 note that training on 80% of areas reproduces the full-sample fit. Loads the
full-data results used in the error-vs-dimension section (AE mean over 10 runs from
`all_stability_results.pkl`, PCA from `pca_rmse_cache.pkl`) and prints their difference from the
CV train means. Print only — not part of the exported table.

In [5]:
with open("../AE_outputs/retraining_stability_500epochs_linscaling/data/all_stability_results.pkl", 'rb') as f:
    _stab = pickle.load(f)
ae_full = np.array([_stab[d]['rmse_mean'] for d in bottleneck_sizes]) * 100

with open("../AE_outputs/engcensus_all/pca_rmse_cache.pkl", 'rb') as f:
    pca_full = np.array(pickle.load(f)) * 100

check = pd.DataFrame({
    'Dimensionality': bottleneck_sizes,
    'AE cv-train':  [v.mean() for v in ae_train],
    'AE full':      ae_full,
    'AE diff':      [v.mean() - fs for v, fs in zip(ae_train, ae_full)],
    'PCA cv-train': [v.mean() for v in pca_train],
    'PCA full':     pca_full,
    'PCA diff':     [v.mean() - fs for v, fs in zip(pca_train, pca_full)],
})
print(check.to_string(index=False, float_format=lambda x: f"{x:+.3f}" if abs(x) < 0.5 else f"{x:.3f}"))
print(f"\nmax |AE diff|  = {check['AE diff'].abs().max():.3f} pp"
      f"\nmax |PCA diff| = {check['PCA diff'].abs().max():.3f} pp")

 Dimensionality  AE cv-train  AE full  AE diff  PCA cv-train  PCA full  PCA diff
              2        3.909    3.875   +0.034         5.445     5.447    -0.002
              4        3.134    3.141   -0.006         4.565     4.566    -0.002
              8        2.511    2.516   -0.005         3.720     3.722    -0.002
             16        2.003    2.006   -0.003         2.813     2.815    -0.002
             32        1.562    1.568   -0.005         2.033     2.035    -0.002
             64        1.105    1.106   -0.000         1.359     1.359    -0.000
            100        0.787    0.794   -0.007         0.944     0.944    +0.001
            128        0.685    0.670   +0.015         0.723     0.723    +0.000

max |AE diff|  = 0.034 pp
max |PCA diff| = 0.002 pp


In [6]:
# relative train→held-out gaps quoted in §3.1 (gap as % of train error; print only)
rel = pd.DataFrame({
    'Dimensionality': bottleneck_sizes,
    'AE gap (pp)':   [t.mean() - tr.mean() for tr, t in zip(ae_train, ae_test)],
    'AE gap (%)':    [100 * (t.mean() - tr.mean()) / tr.mean() for tr, t in zip(ae_train, ae_test)],
    'PCA gap (pp)':  [t.mean() - tr.mean() for tr, t in zip(pca_train, pca_test)],
    'PCA gap (%)':   [100 * (t.mean() - tr.mean()) / tr.mean() for tr, t in zip(pca_train, pca_test)],
})
print(rel.to_string(index=False, float_format=lambda x: f"{x:.3f}" if abs(x) < 0.5 else f"{x:.1f}"))

 Dimensionality  AE gap (pp)  AE gap (%)  PCA gap (pp)  PCA gap (%)
              2        0.161         4.1         0.020        0.374
              4        0.193         6.2         0.015        0.337
              8        0.137         5.5         0.024          0.6
             16        0.081         4.1         0.023          0.8
             32        0.044         2.8         0.022          1.1
             64        0.029         2.6         0.022          1.6
            100        0.025         3.2         0.020          2.1
            128        0.024         3.5         0.022          3.0


## 4. Export for the paper (CSV / HTML / LaTeX)

In [7]:
csv_path  = f"{TABLE_DIR}/cv_2x2_rmse.csv"
html_path = f"{TABLE_DIR}/cv_2x2_rmse.html"
tex_path  = f"{TABLE_DIR}/cv_2x2_rmse.tex"

caption = ("Reconstruction RMSE (%) under 5-fold spatially blocked cross-validation. "
           "Held-out values are the mean (min–max) across the five held-out folds; "
           "train values are the mean across the five training fits, whose fold-to-fold "
           f"spread does not exceed {train_spread_bound:.2f} pp at any dimensionality. "
           "PCA is refit within each fold.")

# CSV
table.to_csv(csv_path, index=False)

# HTML (open directly in Word: File > Open)
head = "".join(f"<th>{c}</th>" for c in table.columns)
body = "".join("<tr>" + "".join(f"<td>{v}</td>" for v in row) + "</tr>"
               for row in table.itertuples(index=False))
html = (
    '<html><head><meta charset="utf-8"><style>'
    'table{border-collapse:collapse;font-family:Calibri,Arial,sans-serif;font-size:11pt}'
    'th,td{border:1px solid #000;padding:4px 10px;text-align:right}'
    'th{background:#f0f0f0}td:first-child,th:first-child{text-align:center}'
    'caption{caption-side:bottom;font-size:9pt;text-align:left;padding-top:6px}</style></head><body>'
    f'<table><caption>{caption}</caption>'
    f'<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'
    '</body></html>'
)
with open(html_path, 'w') as f:
    f.write(html)

# LaTeX (booktabs)
with open(tex_path, 'w') as f:
    f.write(table.to_latex(index=False, escape=False,
                           column_format='c' + 'r' * (table.shape[1] - 1),
                           caption=caption, label='tab:cv'))

print("Wrote:")
for p in (csv_path, html_path, tex_path):
    print("  ", p)

Wrote:
   tables/cv_2x2_rmse.csv
   tables/cv_2x2_rmse.html
   tables/cv_2x2_rmse.tex
